In [4]:
import pandas as pd
import re
import json
import string
from collections import Counter
import nltk
from nltk.corpus import stopwords
import spacy

nltk.download('stopwords')
nlp = spacy.load("en_core_web_sm")

df = pd.read_csv("../data/captions.txt", sep="|")
df.columns = df.columns.str.strip()
print(df.shape)
print(df.columns.tolist())
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ARC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


(40455, 3)
['image_name', 'caption_number', 'caption_text']


,image_name,caption_number,caption_text
0,1000268201_693b08cb0e.jpg,0,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,1,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,2,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,3,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,4,A little girl in a pink dress going into a woo...


In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_caption'] = df['caption_text'].apply(clean_text)
df[['caption_text', 'clean_caption']].head(10)

,caption_text,clean_caption
0,A child in a pink dress is climbing up a set o...,a child in a pink dress is climbing up a set o...
1,A girl going into a wooden building .,a girl going into a wooden building
2,A little girl climbing into a wooden playhouse .,a little girl climbing into a wooden playhouse
3,A little girl climbing the stairs to her playh...,a little girl climbing the stairs to her playh...
4,A little girl in a pink dress going into a woo...,a little girl in a pink dress going into a woo...
5,A black dog and a spotted dog are fighting,a black dog and a spotted dog are fighting
6,A black dog and a tri-colored dog playing with...,a black dog and a tricolored dog playing with ...
7,A black dog and a white dog with brown spots a...,a black dog and a white dog with brown spots a...
8,Two dogs of different breeds looking at each o...,two dogs of different breeds looking at each o...
9,Two dogs on pavement moving toward each other .,two dogs on pavement moving toward each other


In [6]:
def tokenize(text):
    return text.split()

df['tokens'] = df['clean_caption'].apply(tokenize)
df[['clean_caption', 'tokens']].head(10)

,clean_caption,tokens
0,a child in a pink dress is climbing up a set o...,"[a, child, in, a, pink, dress, is, climbing, u..."
1,a girl going into a wooden building,"[a, girl, going, into, a, wooden, building]"
2,a little girl climbing into a wooden playhouse,"[a, little, girl, climbing, into, a, wooden, p..."
3,a little girl climbing the stairs to her playh...,"[a, little, girl, climbing, the, stairs, to, h..."
4,a little girl in a pink dress going into a woo...,"[a, little, girl, in, a, pink, dress, going, i..."
5,a black dog and a spotted dog are fighting,"[a, black, dog, and, a, spotted, dog, are, fig..."
6,a black dog and a tricolored dog playing with ...,"[a, black, dog, and, a, tricolored, dog, playi..."
7,a black dog and a white dog with brown spots a...,"[a, black, dog, and, a, white, dog, with, brow..."
8,two dogs of different breeds looking at each o...,"[two, dogs, of, different, breeds, looking, at..."
9,two dogs on pavement moving toward each other,"[two, dogs, on, pavement, moving, toward, each..."


In [7]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens_no_stopwords'] = df['tokens'].apply(remove_stopwords)
df[['tokens', 'tokens_no_stopwords']].head(10)

,tokens,tokens_no_stopwords
0,"[a, child, in, a, pink, dress, is, climbing, u...","[child, pink, dress, climbing, set, stairs, en..."
1,"[a, girl, going, into, a, wooden, building]","[girl, going, wooden, building]"
2,"[a, little, girl, climbing, into, a, wooden, p...","[little, girl, climbing, wooden, playhouse]"
3,"[a, little, girl, climbing, the, stairs, to, h...","[little, girl, climbing, stairs, playhouse]"
4,"[a, little, girl, in, a, pink, dress, going, i...","[little, girl, pink, dress, going, wooden, cabin]"
5,"[a, black, dog, and, a, spotted, dog, are, fig...","[black, dog, spotted, dog, fighting]"
6,"[a, black, dog, and, a, tricolored, dog, playi...","[black, dog, tricolored, dog, playing, road]"
7,"[a, black, dog, and, a, white, dog, with, brow...","[black, dog, white, dog, brown, spots, staring..."
8,"[two, dogs, of, different, breeds, looking, at...","[two, dogs, different, breeds, looking, road]"
9,"[two, dogs, on, pavement, moving, toward, each...","[two, dogs, pavement, moving, toward]"


In [8]:
def add_start_end_tokens(tokens):
    return ['<start>'] + tokens + ['<end>']

df['final_tokens'] = df['tokens'].apply(add_start_end_tokens)
df[['tokens', 'final_tokens']].head(5)

,tokens,final_tokens
0,"[a, child, in, a, pink, dress, is, climbing, u...","[<start>, a, child, in, a, pink, dress, is, cl..."
1,"[a, girl, going, into, a, wooden, building]","[<start>, a, girl, going, into, a, wooden, bui..."
2,"[a, little, girl, climbing, into, a, wooden, p...","[<start>, a, little, girl, climbing, into, a, ..."
3,"[a, little, girl, climbing, the, stairs, to, h...","[<start>, a, little, girl, climbing, the, stai..."
4,"[a, little, girl, in, a, pink, dress, going, i...","[<start>, a, little, girl, in, a, pink, dress,..."


In [9]:
all_words = []
for tokens in df['final_tokens']:
    all_words.extend(tokens)

word_freq = Counter(all_words)
print(f"Total words: {len(all_words)}")
print(f"Vocabulary size: {len(word_freq)}")

threshold = 2
vocab = [word for word, count in word_freq.items() if count >= threshold]
vocab = ['<pad>', '<unk>'] + vocab

print(f"Final vocabulary size (after threshold filtering): {len(vocab)}")

Total words: 517248
Vocabulary size: 8780
Final vocabulary size (after threshold filtering): 5202


In [10]:
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

print("Sample mappings:", list(word2idx.items())[:10])

Sample mappings: [('<pad>', 0), ('<unk>', 1), ('<start>', 2), ('a', 3), ('child', 4), ('in', 5), ('pink', 6), ('dress', 7), ('is', 8), ('climbing', 9)]


In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)

df[['image_name', 'caption_text', 'clean_caption', 'final_tokens']].to_csv(
    "../data/processed/cleaned_captions.csv", index=False
)

with open("../data/processed/vocab.json", "w") as f:
    json.dump({"word2idx": word2idx, "idx2word": idx2word}, f)

print("✅ Cleaned captions and vocabulary saved to data/processed/")

✅ Cleaned captions and vocabulary saved to data/processed/


In [19]:
import cv2
import numpy as np
from torchvision import transforms
from PIL import Image
import os

In [24]:
image_folder = "../data/Images"
sample_images = df['image_name'].unique()[:100]

channel_issues = []

for img_name in sample_images:
    img_path = os.path.join(image_folder, img_name)
    img = cv2.imread(img_path)
    if img is None:
        channel_issues.append((img_name, "Failed to load"))
        continue
    if len(img.shape) != 3 or img.shape[2] != 3:
        channel_issues.append((img_name, f"Shape: {img.shape}"))

print(f"Images with channel issues: {len(channel_issues)}")
for issue in channel_issues[:10]:
    print(issue)

Images with channel issues: 0


In [25]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img_tensor = preprocess(img)
    return img_tensor

In [26]:
processed_images_folder = "../data/processed/images_resized"
os.makedirs(processed_images_folder, exist_ok=True)

sample_for_processing = df['image_name'].unique()[:50]

for img_name in sample_for_processing:
    img_path = os.path.join(image_folder, img_name)
    try:
        img = Image.open(img_path).convert("RGB")
        img_resized = img.resize((224, 224))
        img_resized.save(os.path.join(processed_images_folder, img_name))
    except Exception as e:
        print(f"Error processing {img_name}: {e}")

print(f"✅ Processed and saved {len(sample_for_processing)} resized images to data/processed/images_resized/")

✅ Processed and saved 50 resized images to data/processed/images_resized/
